# DINOv2-small: manual character identity experiment

Keep this separate from OCR. Use original page pixel coordinates and visually reviewed anonymous identities. The starter set has five sequences from the existing training split: three calibration, two heldout. The project validation/test sets are untouched. These are candidate sequences; replace them if necessary to cover pose and appearance variation. Aim for 50–100 appearances across pages. Draw face, head + hair, and upper-body boxes for the same appearances. Every reported pair stays within one sequence and one crop strategy and crosses pages. Heldout results are independent only if neither their labels nor failures guide threshold or crop-strategy selection.

## 1. Load DINO
Run `download_model.py` first if the local model is missing. Select the existing **AppliedTask (CUDA GPU)** kernel.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if ROOT.name != 'DINO-EXPERMINTATOIN-IDENTITY':
    ROOT = ROOT / 'DINO-EXPERMINTATOIN-IDENTITY'
assert (ROOT / 'identity.py').exists(), ROOT
sys.path.insert(0, str(ROOT))
import identity as dino
processor, model, device = dino.load_model()
print(device, model.config.hidden_size)

## 2. Load manually cropped characters
Open `annotations/annotate.html` in your browser. Drag a character box, enter an anonymous identity such as A/B/C, and save each of the three strategies under the same appearance ID. Labels restart within every sequence. Export and replace `annotations/characters.json` with the downloaded file, then run this cell. Missing strategies are allowed, but fair strategy comparison requires matching appearances. Leave identity blank for ambiguous appearances; those are excluded from labelled pair evaluation.

In [ ]:
annotation_path = ROOT / 'annotations/characters.json'
data, rows = dino.load_annotations(annotation_path)
assert rows, 'Open annotations/annotate.html and annotate characters first.'
rows = dino.export_crops(rows)
print(len(rows), 'crop variants from', len({(r['sequence_id'], r['crop_id']) for r in rows}), 'appearances')
display(FileLink(str(ROOT / 'annotations/annotate.html')))

## 3. Generate normalized embeddings

In [ ]:
vectors = dino.embed(rows, processor, model, device)
print(vectors.shape)
print('L2 norms:', np.linalg.norm(vectors, axis=1).min(), np.linalg.norm(vectors, axis=1).max())

## 4. Calculate similarity matrices

In [ ]:
group_indices = dino.groups(rows)
similarity_matrices = {key: np.clip(vectors[idx] @ vectors[idx].T, -1, 1) for key, idx in group_indices.items()}
list(similarity_matrices)

## 5. Display a similarity matrix

In [ ]:
first_key = next(iter(group_indices))
dino.plot_matrix(rows, vectors, group_indices[first_key])
plt.show()

## 6. Build SAME / DIFFERENT cross-page pairs

In [ ]:
pairs = dino.make_pairs(rows, vectors)
print('Cross-page labelled pairs:', len(pairs))
for strategy in dino.STRATEGIES:
    subset = [p for p in pairs if p['strategy'] == strategy]
    print(strategy, len(subset), 'pairs;', sum(p['same_character'] for p in subset), 'same')

## 7. Plot calibration distributions

In [ ]:
calibrations = {}
for strategy in dino.STRATEGIES:
    subset = [p for p in pairs if p['strategy'] == strategy and p['split'] == 'calibration']
    if {p['same_character'] for p in subset} != {True, False}:
        print(strategy, 'needs labelled same and different cross-page calibration pairs')
        continue
    calibrations[strategy] = dino.calibrate(subset)
    dino.plot_distributions(subset, calibrations[strategy]['threshold'])
    plt.title(strategy + ' — calibration only')
    plt.show()

## 8. Calibrate thresholds and evaluate heldout pairs
Thresholds maximize balanced accuracy on calibration sequences only. Calibration accuracy is a fit score, not a final benchmark. Do not choose a strategy or threshold by heldout performance. Cosine scores span −1 to 1. Different pairs can outnumber same pairs, so report balanced accuracy alongside ordinary accuracy.

In [ ]:
for strategy, report in calibrations.items():
    heldout = [p for p in pairs if p['strategy'] == strategy and p['split'] == 'heldout']
    print(strategy, 'threshold', report['threshold'])
    print('CALIBRATION', report['calibration'])
    print('HELDOUT', dino.metrics(heldout, report['threshold']))

## 9. Cluster within each sequence
Greedy clustering assigns each crop to a normalized mean centroid or creates a new identity. Identities restart per sequence and strategy; assignments are order-sensitive. Pairwise threshold accuracy does not prove clustering accuracy.

In [ ]:
assignments = {}
for strategy, report in calibrations.items():
    assignments.update({i: label for i, label in dino.cluster(rows, vectors, report['threshold']).items() if rows[i]['strategy'] == strategy})
[(rows[i]['sequence_id'], rows[i]['strategy'], rows[i]['crop_id'], label) for i, label in assignments.items()][:20]

## 10. Save results and inspect failures
This reruns extraction to produce a self-contained run directory: embeddings, matrices, pair scores, metrics, failure image grids, annotation snapshot, and separate `character_results.jsonl`. Results contain no OCR text or evaluation identity labels. Use calibration failures to improve the experiment; freeze decisions before inspecting heldout failures.

In [ ]:
from datetime import datetime
output = ROOT / 'outputs' / datetime.now().strftime('%Y%m%d_%H%M%S')
reports = dino.run_experiment(annotation_path, output)
print(json.dumps(reports, indent=2))
print('Saved to', output)
from IPython.display import Image as DisplayImage
for path in sorted(output.glob('*_calibration_failures.png')):
    display(DisplayImage(filename=str(path)))